In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve
import imageio.v2 as imageio
from tqdm import tqdm

# ================= НАСТРОЙКИ =================
L = 80.0
N = 200
h = L / N
x = np.linspace(0, L, N, endpoint=False)

# Маленький шаг для идеальной устойчивости солитона
tau = 0.00005  
T_max = 2.0
steps = int(T_max / tau)
save_every = max(1, steps // 200) # Ровно 200 кадров для плавного GIF

kappa = tau / (h**3)
newton_tol = 1e-9
max_newton_iter = 20

# ================= ТОЧНОЕ РЕШЕНИЕ =================
c = 0.5
x0 = L / 3.0
def soliton_exact(x, t, L):
    x_shifted = (x - x0 - c * t) % L
    x_shifted = np.where(x_shifted > L/2, x_shifted - L, x_shifted) 
    return (c / 2.0) / np.cosh(np.sqrt(c) / 2.0 * x_shifted)**2

u = soliton_exact(x, 0, L)

# ================= БЫСТРЫЙ ВЕКТОРИЗОВАННЫЙ РЕШАТЕЛЬ =================
def solve_kdv_fast(u_prev, kappa, h, tau):
    """
    Решатель КдВ с векторной сборкой 4-диагональной матрицы.
    Работает за O(N) благодаря оптимизированным C-библиотекам scipy.
    """
    u_new = u_prev.copy()
    N_nodes = len(u_new)
    i_idx = np.arange(N_nodes)
    
    for k in range(max_newton_iter):
        # 1. Векторизованные сдвиги (вместо циклов)
        u_ip1 = np.roll(u_new, -1)
        u_im1 = np.roll(u_new, 1)
        u_im2 = np.roll(u_new, 2)
        
        # 2. Невязка F(u)
        L_u = (u_ip1 - 3.0*u_new + 3.0*u_im1 - u_im2) / h**3
        NL_u = (u_ip1**2 - u_im1**2) / h
        F = u_new - u_prev - tau * (L_u - NL_u)
        
        # Проверка сходимости
        if np.max(np.abs(F)) < newton_tol:
            break
        
        # 3. Точный Якобиан (4 диагонали, строго по вашему шаблону)
        diag_0  = 1.0 + 3.0*kappa - (9.0*tau/h)*u_new
        diag_1  = -kappa - (tau/h)*(2.0*u_ip1 + u_new)
        diag_m1 = -3.0*kappa + (12.0*tau/h)*u_im1
        diag_m2 = np.full(N_nodes, kappa)
        
        # 4. Мгновенная векторная сборка матрицы в COO-формате
        # Это заменяет медленные циклы и .tolil()
        rows = np.concatenate([i_idx, i_idx, i_idx, i_idx])
        cols = np.concatenate([
            i_idx, 
            (i_idx + 1) % N_nodes, 
            (i_idx - 1) % N_nodes, 
            (i_idx - 2) % N_nodes
        ])
        data = np.concatenate([diag_0, diag_1, diag_m1, diag_m2])
        
        J = coo_matrix((data, (rows, cols)), shape=(N_nodes, N_nodes)).tocsc()
        
        # 5. Решение СЛАУ (в C, с автоматическим выбором главного элемента, без деления на 0)
        delta_u = spsolve(J, -F)
        u_new += delta_u
        
    return u_new

# ================= ЗАПУСК И СОХРАНЕНИЕ В GIF =================
print(f"Расчёт: {steps} шагов, tau={tau}, сохранение каждые {save_every} шагов")
print("Используется векторизованная 4-диагональная схема (O(N), стабильная)")

frames = []
fig, ax = plt.subplots(figsize=(10, 4))
line, = ax.plot([], [], 'b-', lw=2)
ax.set_xlim(0, L)
ax.set_ylim(-0.1, np.max(u) * 1.2)
ax.set_xlabel('x')
ax.set_ylabel('u(x,t)')
ax.set_title('Солитон КдВ (Векторизованная 4-диагональная схема)')
ax.grid(True, alpha=0.3)
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)
plt.tight_layout()

for step in tqdm(range(steps), desc="KdV Solver"):
    u = solve_kdv_fast(u, kappa, h, tau)
    
    if step % save_every == 0:
        line.set_ydata(u)
        line.set_xdata(x)
        time_text.set_text(f't = {(step + 1) * tau:.4f}')
        
        # Современный способ получения RGB-массива (совместим с Matplotlib >= 3.6)
        fig.canvas.draw()
        rgba_array = np.array(fig.canvas.renderer.buffer_rgba())
        img = rgba_array[:, :, :3] # Берём только RGB каналы
        frames.append(img)

plt.close(fig)

print("Сохраняем GIF...")
imageio.mimsave('kdv_soliton_fast.gif', frames, fps=30, loop=0)
print("Готово! Файл kdv_soliton_fast.gif сохранён в текущей папке.")

# Финальная проверка
print(f"\nПроверка периодичности:")
print(f"u[0]   = {u[0]:.8f}")
print(f"u[N-1] = {u[-1]:.8f}")
print(f"Разница: {abs(u[0] - u[-1]):.2e}")

Расчёт: 40000 шагов, tau=5e-05, сохранение каждые 200 шагов
Используется векторизованная 4-диагональная схема (O(N), стабильная)


KdV Solver: 100%|██████████| 40000/40000 [00:15<00:00, 2565.70it/s]


Сохраняем GIF...
Готово! Файл kdv_soliton_fast.gif сохранён в текущей папке.

Проверка периодичности:
u[0]   = 0.00000001
u[N-1] = 0.00000001
Разница: 2.96e-09
